# 🚗 Treinamento do YOLO com o Dataset Oficial UFPR-ALPR (Placas Brasileiras)
### Dataset: UFPR-ALPR (Universidade Federal do Paraná - 4.500 imagens anotadas de veículos brasileiros)

Este notebook automatiza:
1. Download direto do dataset oficial `yj4Iu2-UFPR-ALPR.zip`
2. Descompactação e conversão das anotações de placas e caracteres para o padrão YOLO
3. Treinamento do modelo YOLO com GPU T4 (Transfer Learning)
4. Avaliação com métricas de validação em placas brasileiras (Mercosul e Antigas)
5. Download dos pesos treinados (`best.pt`) para o seu projeto local

## 1. Verificação da GPU
Certifique-se de que a GPU está ativada no Colab em: **Ambiente de execução > Alterar tipo de ambiente de execução > GPU T4**.

In [ ]:
!nvidia-smi

## 2. Instalação das Dependências

In [ ]:
!pip install -q ultralytics pillow
print("✅ Ultralytics instalado com sucesso!")

## 3. Download e Descompactação do Dataset UFPR-ALPR
Baixamos diretamente o arquivo compactado oficial da UFPR.

In [ ]:
import os
import zipfile
import urllib.request

URL_DATASET = "https://www.inf.ufpr.br/vri/databases/yj4Iu2-UFPR-ALPR.zip"
ZIP_DESTINO = "/content/UFPR-ALPR.zip"
PASTA_EXTRACAO = "/content/ufpr_raw"

if not os.path.exists(ZIP_DESTINO):
    print("📥 Baixando dataset oficial UFPR-ALPR (aguarde alguns instantes)...")
    headers = {'User-Agent': 'Mozilla/5.0'}
    req = urllib.request.Request(URL_DATASET, headers=headers)
    with urllib.request.urlopen(req) as resp, open(ZIP_DESTINO, 'wb') as f:
        f.write(resp.read())
    print("✅ Download do ZIP concluído!")

print("📦 Descompactando arquivos...")
with zipfile.ZipFile(ZIP_DESTINO, 'r') as zip_ref:
    zip_ref.extractall(PASTA_EXTRACAO)

print("✅ Descompactação finalizada com sucesso!")

## 4. Conversão das Anotações da UFPR para o Formato YOLO
O UFPR-ALPR armazena anotações em arquivos `.txt` contendo `position_plate: X Y W H` e as classes dos caracteres. Convertemos as coordenadas para valores normalizados `(x_center, y_center, width, height)` e dividimos em Treino, Validação e Teste.

In [ ]:
import glob
import shutil
import cv2

DATASET_YOLO = "/content/dataset_ufpr_yolo"
os.makedirs(os.path.join(DATASET_YOLO, "train/images"), exist_ok=True)
os.makedirs(os.path.join(DATASET_YOLO, "train/labels"), exist_ok=True)
os.makedirs(os.path.join(DATASET_YOLO, "val/images"), exist_ok=True)
os.makedirs(os.path.join(DATASET_YOLO, "val/labels"), exist_ok=True)

def processar_split(pasta_split, split_destino):
    arquivos_txt = glob.glob(os.path.join(pasta_split, "**/*.txt"), recursive=True)
    total = 0
    for txt_path in arquivos_txt:
        img_path = txt_path.replace(".txt", ".png")
        if not os.path.exists(img_path):
            img_path = txt_path.replace(".txt", ".jpg")
            if not os.path.exists(img_path):
                continue
        
        # Lê dimensões da imagem
        img = cv2.imread(img_path)
        if img is None:
            continue
        h_img, w_img = img.shape[:2]
        
        # Lê anotação de posição da placa
        with open(txt_path, 'r', encoding='utf-8', errors='ignore') as f:
            linhas = f.readlines()
            
        yolo_labels = []
        for linha in linhas:
            if "position_plate:" in linha:
                partes = linha.strip().split(":")[1].strip().split()
                x, y, w, h = map(float, partes[:4])
                # Normaliza para YOLO
                x_center = (x + w / 2.0) / w_img
                y_center = (y + h / 2.0) / h_img
                norm_w = w / w_img
                norm_h = h / h_img
                yolo_labels.append(f"0 {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}")
                
        if yolo_labels:
            nome_base = os.path.basename(img_path)
            shutil.copy(img_path, os.path.join(DATASET_YOLO, f"{split_destino}/images", nome_base))
            
            txt_salvo = os.path.join(DATASET_YOLO, f"{split_destino}/labels", nome_base.rsplit('.', 1)[0] + ".txt")
            with open(txt_salvo, 'w') as f_out:
                f_out.write("\n".join(yolo_labels))
            total += 1
            
    print(f"Split {split_destino}: {total} imagens convertidas.")

# Procura as pastas de treino e validação descompactadas
pastas_encontradas = glob.glob(os.path.join(PASTA_EXTRACAO, "**"), recursive=True)
for p in pastas_encontradas:
    if os.path.isdir(p):
        if "training" in p.lower() or "train" in p.lower():
            processar_split(p, "train")
        elif "validation" in p.lower() or "val" in p.lower() or "testing" in p.lower():
            processar_split(p, "val")

# Cria arquivo data.yaml
yaml_content = f"""
path: {DATASET_YOLO}
train: train/images
val: val/images

names:
  0: placa
"""

yaml_path = os.path.join(DATASET_YOLO, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content.strip())

print(f"✅ data.yaml configurado com sucesso em: {yaml_path}")

## 5. Treinamento do YOLO com Dataset Brasileiro UFPR-ALPR

In [ ]:
from ultralytics import YOLO

# Carrega modelo base YOLO
modelo = YOLO("yolov8n.pt")

# Treinamento com GPU T4
resultados = modelo.train(
    data=os.path.join(DATASET_YOLO, "data.yaml"),
    epochs=40,
    imgsz=640,
    batch=16,
    patience=10,
    save=True,
    name="yolo_ufpr_placas_brasil"
)

print("🎉 Treinamento com UFPR-ALPR finalizado com sucesso!")

## 6. Avaliação das Métricas e Desempenho

In [ ]:
from IPython.display import Image, display
import glob

for grafico in glob.glob("runs/detect/yolo_ufpr_placas_brasil/*.png"):
    print(f"📊 Gráfico: {grafico}")
    display(Image(filename=grafico))
    print("-" * 50)

## 7. Download dos Melhores Pesos (`best.pt`)
Baixe o arquivo `best.pt` e coloque-o na pasta `models/best.pt` do seu projeto local.

In [ ]:
from google.colab import files

caminho_pesos = "runs/detect/yolo_ufpr_placas_brasil/weights/best.pt"
if os.path.exists(caminho_pesos):
    print("⬇️ Baixando modelo treinado com UFPR-ALPR...")
    files.download(caminho_pesos)
else:
    print("❌ Arquivo best.pt não encontrado.")